# Clase 45 — Regresión logística

Notebook introductorio para **clasificar** (predecir una clase 0/1) con scikit-learn.

**Caso:** tumores de mama del dataset `load_breast_cancer` (scikit-learn). Objetivo: estimar si el diagnóstico es **maligno (0)** o **benigno (1)**.

**Diferencia con la Clase 44:** la regresión lineal predice un **número continuo**; la logística predice una **categoría**.

**Cómo usarlo:** ejecutá las celdas en orden.

### Objetivos

1. Entender **por qué** transformamos y seleccionamos pocas columnas.
2. Separar train/test con `stratify` para respetar el balance de clases.
3. Entrenar `LogisticRegression`.
4. Evaluar con accuracy, matriz de confusión y classification report.
5. Guardar y recargar el modelo con `joblib`.

### Requisitos

- Entorno **`.venv`** con `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`, `joblib`.
- Kernel **Python (.venv)**.


## Paso 0 — Importar librerías


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

sns.set_theme(style="whitegrid")


## Paso 1 — Leer dataset

Cargamos Breast Cancer desde scikit-learn (no hace falta un CSV externo).


In [ ]:
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["diagnostico"] = data.target  # 0 = maligno, 1 = benigno

print("Filas:", len(df), "| Columnas:", len(df.columns))
print("Clases:", dict(zip(data.target_names, np.bincount(data.target))))
print("  (en target: 0 = malignant, 1 = benign)")
df.head()


In [ ]:
df.info()
df[["mean radius", "mean texture", "mean perimeter", "diagnostico"]].describe()


## Paso 2 — Por qué transformar

El dataset trae **30 features**. Para principiantes usamos solo **3**, con nombres claros.

| Situación | Qué hacemos | Por qué |
|-----------|-------------|---------|
| Nombres con espacios (`mean radius`) | `rename` a snake_case | Evita errores al tipiar y al graficar |
| Demasiadas columnas | Elegir 3 predictores | El modelo se entiende mejor en clase |
| Target poco explícito | Columna `diagnostico` | Dejamos claro qué clasificamos |
| Clases desbalanceadas posibles | `stratify` en el split | Train y test mantienen la misma proporción |

> Internamente la regresión logística usa una curva en forma de **S** (sigmoide) para convertir un puntaje en probabilidad entre 0 y 1. Luego elige la clase más probable.


### Paso 2.1 — Transformar y seleccionar columnas


In [ ]:
df = df.rename(columns={
    "mean radius": "radio_medio",
    "mean texture": "textura_media",
    "mean perimeter": "perimetro_medio",
})

df = df[["radio_medio", "textura_media", "perimetro_medio", "diagnostico"]]
df.head()


## Paso 3 — Separar train y test

Usamos `stratify=y` para que train y test tengan una proporción similar de maligno/benigno.


In [ ]:
X = df[["radio_medio", "textura_media", "perimetro_medio"]]
y = df["diagnostico"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", len(X_train), "filas | proporción clase 1:", round(y_train.mean(), 3))
print("Test :", len(X_test), "filas | proporción clase 1:", round(y_test.mean(), 3))


## Paso 4 — Exploración en train

Miramos el balance de clases y cómo se separan las clases en una variable.


In [ ]:
print("Conteo de clases en train:")
print(y_train.value_counts().rename({0: "maligno (0)", 1: "benigno (1)"}))

plt.figure(figsize=(7, 5))
sns.histplot(
    data=pd.concat([X_train, y_train], axis=1),
    x="radio_medio",
    hue="diagnostico",
    bins=20,
    kde=True,
)
plt.title("Train: radio_medio por diagnóstico")
plt.show()


## Paso 5 — Entrenar el modelo

`LogisticRegression` aprende pesos para cada feature y estima la probabilidad de cada clase.


In [ ]:
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

print("Intercepto:", round(modelo.intercept_[0], 3))
print("Coeficientes:")
for nombre, coef in zip(X_train.columns, modelo.coef_[0]):
    print(f"  {nombre}: {coef:.3f}")


## Paso 6 — Evaluar en test

| Métrica | Qué mide (idea simple) |
|---------|------------------------|
| **Accuracy** | Proporción de aciertos totales |
| **Precision** | De lo que predije como positivo, cuánto era correcto |
| **Recall** | De los positivos reales, cuántos encontré |
| **Matriz de confusión** | Acierto/error por clase |


In [ ]:
y_pred = modelo.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["maligno", "benigno"]))


In [ ]:
matriz = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=["maligno", "benigno"])
disp.plot(cmap="Blues")
plt.title("Matriz de confusión (test)")
plt.show()


## Paso 7 — Guardar el modelo


In [ ]:
Path("modelos").mkdir(exist_ok=True)
ruta = Path("modelos") / "regresion_logistica_cancer.joblib"

joblib.dump(modelo, ruta)
print("Modelo guardado en:", ruta)


In [ ]:
# Cargar y predecir un ejemplo
modelo_cargado = joblib.load(ruta)

ejemplo = pd.DataFrame({
    "radio_medio": [14.0],
    "textura_media": [20.0],
    "perimetro_medio": [90.0],
})

pred = modelo_cargado.predict(ejemplo)[0]
proba = modelo_cargado.predict_proba(ejemplo)[0]

print("Clase predicha:", pred, "(0=maligno, 1=benigno)")
print("Probabilidades:", {"maligno": round(proba[0], 3), "benigno": round(proba[1], 3)})


## Cierre — Lineal vs logística

| | Regresión lineal (Clase 44) | Regresión logística (Clase 45) |
|--|-----------------------------|-------------------------------|
| Objetivo | Número continuo | Clase (0/1) |
| Ejemplo | Mortalidad | Maligno / benigno |
| Métricas típicas | MAE, RMSE, R² | Accuracy, matriz, precision/recall |
| Guardado | `joblib.dump` | `joblib.dump` |

Checklist del flujo (igual que en lineal):

1. Cargar → 2. Transformar → 3. Split → 4. Entrenar → 5. Evaluar → 6. Guardar
